<a href="https://colab.research.google.com/github/OluwatifeAkinsola/RAG-Chatbot/blob/main/LOCAL_RAG_CHATBOT_using_Phi_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install required libraries


In [ ]:
#Step 1: Install required libraries
pip install langchain langchain-community chromadb sentence-transformers pypdf ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

Step2: Download a small open-source embedding model

In [ ]:
#Download a small open-source embedding model
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#Load and Chunk a PDF
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('/Neural Networks from Scratch in Python.pdf')
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size =500, chunk_overlap =50)
chunks = splitter.split_documents(documents)

In [ ]:
#Create embeddings and store in ChromaDB
import chromadb
from chromadb.utils import embedding_functions

#use the same embedding model
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name ="all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path ="./chroma_db")
collection = client.get_or_create_collection(name = "my_docs", embedding_function=embedding_fn)

#Add chunks to collection
for i, chunk in enumerate(chunks):
    collection.add(
       documents = [chunk.page_content],
       metadatas =[{"source": chunk.metadata.get("source", "unknown")}],
       ids = [str(i)] )

In [ ]:
#Step 5: Retrieve relevant chunks for a query
def retrieve(query, k=3):
    results = collection.query(query_texts=[query], n_results=k)
    return results['documents'][0]  # list of chunk texts

In [ ]:
# First, install the required library
!pip install transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

# Define your lightweight model
model_name = "microsoft/phi-2"
print("Loading the lightweight model...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Set the pad_token_id to the eos_token_id if it's not already set for the tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model configuration and explicitly set pad_token_id
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(model_name, config=config, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
print("Model loaded!")

def ask_llm(question, context_chunks):
    if not context_chunks:
        return "I couldn't find relevant information in the document to answer that."

    context = "\n\n".join(context_chunks)
    # A clear and direct prompt
    prompt = f"""You are a precise question-answering assistant. Use only the provided context to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part after "Answer:"
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()
    return answer

Loading the lightweight model...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [ ]:
def chat_loop():
    print("My Smart Assistant is ready!")
    while True:
        query = input("\nYou: ")
        if query.lower() == 'quit':
            break
        # This is the core RAG step!
        relevant_chunks = retrieve(query, k=3)
        print("\nMy Assistant found these relevant quotes:\n")
        for i, chunk in enumerate(relevant_chunks):
            print(f"--- QUOTE {i+1} ---")
            print(chunk)
            print("-" * 20)

In [ ]:
pip install gradio

In [ ]:
import gradio as gr

def answer_fn(question):
    chunks = retrieve(question)
    return ask_llm(question, chunks)

iface = gr.Interface(fn=answer_fn, inputs="text", outputs="text", title="Local RAG Chatbot")
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fec6f1771735cc140a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
